# Phase 9: Gemma4 LoRA Fine-tune (Unsloth + GRPO)

**Base model**: `gemma4-rotorquant:latest` (IQ4_XS, 5.3GB)  
**Dataset**: `lora-training-pairs-contrastive.jsonl` (99 pairs: correct/wrong_tool/degraded)  
**Method**: GRPO with 7 reward functions  
**Target hardware**: Colab G4 Blackwell 96GB  

Upload `memory/packets/lora-training-pairs-contrastive.jsonl` to this session before running.

In [ ]:
# Install Unsloth (Colab-specific)
!pip install unsloth --quiet
!pip install trl datasets transformers accelerate bitsandbytes --quiet

In [ ]:
import json, re
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
import torch

MAX_SEQ_LENGTH = 1024
DTYPE = torch.bfloat16
LOAD_IN_4BIT = True

# Load base model — use Gemma 4 2B as Colab-friendly proxy
# For production: swap for the local gemma4-rotorquant GGUF via llama.cpp adapter
MODEL_ID = 'google/gemma-2-2b-it'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

In [ ]:
# Load training pairs
with open('lora-training-pairs-contrastive.jsonl') as f:
    raw = [json.loads(l) for l in f if l.strip()]

print(f'Loaded {len(raw)} pairs')
by_type = {}
for r in raw:
    t = r['metadata']['trace_type']
    by_type[t] = by_type.get(t, 0) + 1
print('by type:', by_type)

SYSTEM_PROMPT = """You are a tool-routing agent for a legal AI platform.
Given a query route, context, and source references, select the correct retrieval tool.
Available tools: trace.kag_search, trace.qdrant_search, trace.neo4j_query, trace.redis_get,
trace.postgres_query, trace.file_read, trace.context_assemble, trace.embed,
mcp.unified_ast_query, mcp.agentic_recommendation, mcp.system_health_check,
ace.route_packet, ace.feature_lookup"""

def format_prompt(pair):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': pair['instruction'] + '\n' + pair['input']},
    ]

def format_example(pair):
    return {
        'prompt': tokenizer.apply_chat_template(format_prompt(pair), tokenize=False, add_generation_prompt=True),
        'completion': pair['output'],
        'reward': pair['metadata']['reward'],
        'trace_type': pair['metadata']['trace_type'],
    }

dataset = Dataset.from_list([format_example(p) for p in raw])
print(dataset)

In [ ]:
# 7 GRPO reward functions

VALID_TOOLS = {
    'trace.kag_search', 'trace.qdrant_search', 'trace.neo4j_query', 'trace.redis_get',
    'trace.postgres_query', 'trace.file_read', 'trace.context_assemble', 'trace.embed',
    'mcp.unified_ast_query', 'mcp.agentic_recommendation', 'mcp.system_health_check',
    'ace.route_packet', 'ace.feature_lookup',
}

def extract_tool(text):
    m = re.search(r'Tool:\s*(\S+)', text)
    return m.group(1).strip() if m else None

def r1_valid_tool(completions, **kwargs):
    """R1: Output contains a valid tool name"""
    return [1.0 if extract_tool(c) in VALID_TOOLS else 0.0 for c in completions]

def r2_has_reason(completions, **kwargs):
    """R2: Output contains a Reason line"""
    return [1.0 if 'Reason:' in c else 0.0 for c in completions]

def r3_has_reward(completions, **kwargs):
    """R3: Output contains a Reward line"""
    return [1.0 if 'Reward:' in c else 0.0 for c in completions]

def r4_correct_tool(completions, reward, **kwargs):
    """R4: Tool matches ground truth (reward=0.9 traces only)"""
    scores = []
    for c, r in zip(completions, reward):
        if r >= 0.9:
            tool = extract_tool(c)
            scores.append(1.0 if tool in VALID_TOOLS else 0.0)
        else:
            scores.append(0.5)  # neutral for wrong/degraded
    return scores

def r5_format_correct(completions, **kwargs):
    """R5: Output has Tool/Reason/Reward in order"""
    def check(c):
        ti = c.find('Tool:')
        ri = c.find('Reason:')
        rw = c.find('Reward:')
        return 1.0 if 0 <= ti < ri < rw else 0.0
    return [check(c) for c in completions]

def r6_brevity(completions, **kwargs):
    """R6: Response is concise (<200 chars)"""
    return [1.0 if len(c) < 200 else max(0.0, 1.0 - (len(c)-200)/500) for c in completions]

def r7_no_hallucination(completions, **kwargs):
    """R7: No invented tool names (not in VALID_TOOLS but looks like a tool)"""
    def check(c):
        tool = extract_tool(c)
        if tool is None: return 0.5
        if tool in VALID_TOOLS: return 1.0
        # Penalize things that look like tool names but aren't
        if '.' in tool or ':' in tool: return 0.0
        return 0.5
    return [check(c) for c in completions]

print('7 reward functions defined')

In [ ]:
# GRPO training config
training_args = GRPOConfig(
    output_dir='./lora-output',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_generations=6,           # 6 completions per prompt
    max_prompt_length=512,
    max_completion_length=128,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    bf16=True,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        r1_valid_tool,
        r2_has_reason,
        r3_has_reward,
        r4_correct_tool,
        r5_format_correct,
        r6_brevity,
        r7_no_hallucination,
    ],
    args=training_args,
    train_dataset=dataset,
)

print('Trainer ready. Starting GRPO...')
trainer.train()

In [ ]:
# Save adapter + merge
model.save_pretrained('./lora-output/adapter')
tokenizer.save_pretrained('./lora-output/adapter')
print('Adapter saved to ./lora-output/adapter')

# Merge and export to GGUF for Ollama
# model.save_pretrained_merged('./lora-output/merged', tokenizer, save_method='merged_16bit')
# Then: llama.cpp/convert_hf_to_gguf.py ./lora-output/merged --outtype q4_K_M
print('\nTo push to Ollama registry:')
print('  ollama create gemma4-legal-routing:latest -f Modelfile')
print('  (Modelfile FROM ./lora-output/adapter + SYSTEM prompt)')

In [ ]:
# Quick eval on training set
FastLanguageModel.for_inference(model)

correct = 0
for pair in raw[:10]:
    inputs = tokenizer(
        tokenizer.apply_chat_template(format_prompt(pair), tokenize=False, add_generation_prompt=True),
        return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    generated = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    tool = extract_tool(generated)
    expected_tool = extract_tool(pair['output'])
    match = tool == expected_tool
    if match: correct += 1
    print(f'  expected={expected_tool} got={tool} {"✓" if match else "✗"}')

print(f'\nAccuracy on first 10: {correct}/10')